In [7]:
import pandas as pd
from pathlib import Path


In [15]:
class Song:
    def __init__(self, title, artist, year):
        """
        Konstruktor koji postavlja osnovne atribute pesme.
        """
        self.title = title
        self.artist = artist
        self.year = year

    def __str__(self):
        """
        Vraća čitljiv tekstualni opis pesme u formatu: <title> - <artist> (<year>)
        """
        return f"<{self.title}>-<{self.artist}>({self.year})"
    @classmethod
    def from_song_data(cls, title, songs_df):
        """
        Alternativni konstruktor koji prima DataFrame i naziv pesme,
        pronalazi red i vraća instanciran objekat klase Song.
        """
        pesma_red = songs_df[songs_df['track'] == title]

        if pesma_red.empty:
            return None

        red_podaci = pesma_red.iloc[0]
        formatiran_datum = str(red_podaci['release_date']).strip()

        # Ekstrakcija godine iz različitih formata datuma
        if '/' in formatiran_datum:
            godina = int(formatiran_datum.split('/')[-1])
        elif '-' in formatiran_datum:
            godina = int(formatiran_datum.split('-')[0])
        else:
            godina = int(formatiran_datum)

        return cls(red_podaci['track'], red_podaci['artist'], godina)

In [14]:
# --- TESTIRANJE KLASE (Zasebna ćelija) ---
# Ručno uneti podaci za testiranje
test_pesma = Song("Our House", "Crosby, Stills, Nash & Young", 1970)
print("Manuelni test klase Song:")
print(str(test_pesma))

Manuelni test klase Song:
<Our House>-<Crosby, Stills, Nash & Young>(1970)


In [17]:
# Učitavanje dataset-a iz fajla csny.csv koristeći pandas i pathlib
WORKING_DIR = Path.cwd() / 'data'
putanja_do_fajla = WORKING_DIR / "csny.csv"
csny_songs = pd.read_csv(putanja_do_fajla)

# Prikaz prvih nekoliko redova radi provere (opciono)
print(f"Dataset uspešno učitan. Ukupno redova: {len(csny_songs)}")
csny_songs.head(3)

Dataset uspešno učitan. Ukupno redova: 774


,track,album,album_type,artist,release_date,recording_period,recording_location,album_duration,album_genre,album_styles,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,Chicago (CSNY),4 Way Street,live,"Crosby, Stills, Nash & Young",7/4/1971,NaN,NaN,1:49:44,Pop/Rock,"Contemporary Pop/Rock, Singer/Songwriter, Albu...",...,9,-10.210,0,0.0330,0.912,0.000017,0.866,0.405,147.037,4
1,Find the Cost of Freedom,4 Way Street,live,"Crosby, Stills, Nash & Young",7/4/1971,NaN,NaN,1:49:44,Pop/Rock,"Contemporary Pop/Rock, Singer/Songwriter, Albu...",...,8,-18.311,1,0.0348,0.714,0.000010,0.270,0.575,75.233,4
2,King Midas in Reverse,4 Way Street,live,"Crosby, Stills, Nash & Young",7/4/1971,NaN,NaN,1:49:44,Pop/Rock,"Contemporary Pop/Rock, Singer/Songwriter, Albu...",...,1,-13.016,1,0.2800,0.689,0.000000,0.702,0.464,155.204,4


In [22]:
def song(title, songs_df:pd.DataFrame):
    """
    Funkcija izdvaja podatke o pesmi sa prosleđenim naslovom,
    rešava problem različitih formata datuma i kreira objekat klase Song.
    """
    # Izdvajanje reda koji odgovara nazivu pesme (kolona 'track' u csny.csv)
    pesma_red = songs_df[songs_df['track'] == title]

    # Neregularan slučaj: ako pesma ne postoji u dataset-u
    if pesma_red.empty:
        print(f"Neregularan slučaj: Pesma sa nazivom '{title}' ne postoji u dataset-u.")
        return None

    # Regularan slučaj: uzimamo podatke iz prvog pronađenog reda
    red_podaci = pesma_red.iloc[0]
    formatiran_datum = str(red_podaci['release_date']).strip()

    # Pokrivanje svih formata datuma iz csny.csv za ekstrakciju godine
    if '/' in formatiran_datum:
        godina = int(formatiran_datum.split('/')[-1])
    elif '-' in formatiran_datum:
        godina = int(formatiran_datum.split('-')[0])
    else:
        godina = int(formatiran_datum)

    # Kreiranje i testiranje objekta klase Song
    kreirana_pesma = Song(red_podaci['track'], red_podaci['artist'], godina)
    return kreirana_pesma

# --- TESTIRANJE FUNKCIJE SONG (Zasebna ćelija) ---
print("Testiranje funkcije 'song':")

# Testiranje regularnih slučajeva (pesme koje postoje u bazi)
pesma_reg1 = song("Chicago (CSNY)", csny_songs)
if pesma_reg1:
    print(f"Ispis regularne pesme 1: {pesma_reg1}")

pesma_reg2 = song("Ohio (CSNY)", csny_songs)
if pesma_reg2:
    print(f"Ispis regularne pesme 2: {pesma_reg2}")

print("-" * 40)

# Testiranje neregularnog slučaja
pesma_nereg = song("Nepostojeća Pesma 123", csny_songs)

Testiranje funkcije 'song':
Ispis regularne pesme 1: <Chicago (CSNY)>-<Crosby, Stills, Nash & Young>(1971)
Ispis regularne pesme 2: <Ohio (CSNY)>-<Crosby, Stills, Nash & Young>(1971)
----------------------------------------
Neregularan slučaj: Pesma sa nazivom 'Nepostojeća Pesma 123' ne postoji u dataset-u.


In [12]:
# --- TESTIRANJE ALTERNATIVNOG KONSTRUKTORA (Zasebna ćelija) ---
print("Testiranje alternativnog konstruktora nad nasumičnim redovima:")

# Uzimanje 3 nasumična naziva pesama iz učitanog dataset-a
nasumicni_naslovi = csny_songs['track'].sample(3).tolist()

for naslov in nasumicni_naslovi:
    pesma_objekat = Song.from_song_data(naslov, csny_songs)
    if pesma_objekat:
        print(f"Uspešno instancirano: {pesma_objekat}")

Testiranje alternativnog konstruktora nad nasumičnim redovima:
Uspešno instancirano: <Hero>-<David Crosby>(1993)
Uspešno instancirano: <Prairie Wind>-<Neil Young>(2005)
Uspešno instancirano: <Pocahontas>-<Neil Young>(2017)


In [13]:
import random

def songs(songs_df):
    """
    Funkcija prolazi kroz ceo dataset, kreira objekat klase Song
    za svaku pesmu i vraća ih u listi song_list.
    """
    song_list = []

    for _, red_podaci in songs_df.iterrows():
        formatiran_datum = str(red_podaci['release_date']).strip()

        # Ekstrakcija godine
        if '/' in formatiran_datum:
            godina = int(formatiran_datum.split('/')[-1])
        elif '-' in formatiran_datum:
            godina = int(formatiran_datum.split('-')[0])
        else:
            godina = int(formatiran_datum)

        # Kreiranje objekta i dodavanje u listu
        nova_pesma = Song(red_podaci['track'], red_podaci['artist'], godina)
        song_list.append(nova_pesma)

    return song_list

# --- TESTIRANJE FUNKCIJE SONGS (Zasebna ćelija) ---
# Pozivanje funkcije nad celim datasetom
sve_pesme_lista = songs(csny_songs)
print(f"Ukupno kreirano objekata u listi: {len(sve_pesme_lista)}")
print("-" * 40)

# Nasumično biranje nekoliko pesama iz rezultujuće liste radi demonstracije __str__() metode
izabrane_nasumicne = random.sample(sve_pesme_lista, 4)

print("Demonstracija rada __str__() metode za nasumično izabrane pesme iz liste:")
for p in izabrane_nasumicne:
    print(p)

Ukupno kreirano objekata u listi: 774
----------------------------------------
Demonstracija rada __str__() metode za nasumično izabrane pesme iz liste:
<The Bridge>-<Neil Young>(1973)
<Everybody's Talking>-<Stephen Stills>(1978)
<Nighttime for Generals>-<Crosby, Stills, Nash & Young>(1988)
<Tomorrow Is Another Day>-<Crosby, Stills & Nash>(1982)
